# Entrenamiento del detector de puntos clave de la pista

Ajusta una ResNet50 para predecir los 14 puntos clave de la pista (28 coordenadas).
Dataset: [TennisCourtDetector](https://github.com/yastrebksv/TennisCourtDetector).

Al terminar, copia `keypoints_model.pth` a `model/models/keypoints_model.pth`.

In [ ]:
# Dataset público de TennisCourtDetector (Google Drive).
!pip install gdown
!gdown 1lhAaeQCmk2y440PmagA0KmIVBIysVMwu -O tennis_court_det_dataset.zip

In [ ]:
!unzip -q tennis_court_det_dataset.zip

## Configuración

In [ ]:
import json

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

IMAGE_SIZE = 224
NUM_KEYPOINTS = 14
EPOCHS = 20
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
CHECKPOINT_PATH = "keypoints_model.pth"

## Dataset

In [ ]:
class KeypointsDataset(Dataset):
    """Devuelve la imagen a 224x224, los puntos escalados a 224 y el tamaño original (ancho, alto)."""

    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file) as f:
            self.data = json.load(f)

        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
        if img is None:
            raise FileNotFoundError(f"No se pudo leer la imagen {item['id']}.png")
        h, w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(img)
        kps = np.array(item["kps"], dtype=np.float32).flatten()

        kps[::2] *= IMAGE_SIZE / w  # coordenadas x
        kps[1::2] *= IMAGE_SIZE / h  # coordenadas y

        return img, kps, torch.tensor([w, h], dtype=torch.float32)

In [ ]:
train_dataset = KeypointsDataset("data/images", "data/data_train.json")
val_dataset = KeypointsDataset("data/images", "data/data_val.json")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Entrenamiento: {len(train_dataset)} imágenes, validación: {len(val_dataset)}")

## Modelo

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Linear(model.fc.in_features, NUM_KEYPOINTS * 2)  # sustituye la última capa
model = model.to(device)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Validación

Error medio en píxeles **de la imagen original**: distancia euclídea entre cada punto predicho y el real,
promediada sobre los 14 puntos y todas las imágenes de validación.

In [ ]:
def mean_pixel_error(model, loader):
    model.eval()
    total_error, total_points = 0.0, 0
    with torch.no_grad():
        for imgs, kps, sizes in loader:
            outputs = model(imgs.to(device)).cpu()
            scale = (sizes / IMAGE_SIZE).unsqueeze(1)  # (lote, 1, 2): factor x e y de cada imagen
            pred = outputs.view(-1, NUM_KEYPOINTS, 2) * scale
            true = kps.view(-1, NUM_KEYPOINTS, 2) * scale
            total_error += torch.linalg.norm(pred - true, dim=2).sum().item()
            total_points += pred.shape[0] * NUM_KEYPOINTS
    return total_error / total_points

## Entrenamiento (se guarda el modelo con menor error de validación)

In [ ]:
best_error = float("inf")
for epoch in range(EPOCHS):
    model.train()
    for i, (imgs, kps, _) in enumerate(train_loader):
        imgs = imgs.to(device)
        kps = kps.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, kps)
        loss.backward()
        optimizer.step()

        if i % 10 == 0:
            print(f"Época {epoch}, iteración {i}, loss: {loss.item():.4f}")

    val_error = mean_pixel_error(model, val_loader)
    print(f"Época {epoch}: error medio de validación = {val_error:.2f} px")
    if val_error < best_error:
        best_error = val_error
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  nuevo mejor modelo guardado en {CHECKPOINT_PATH}")

print(f"Mejor error medio de validación: {best_error:.2f} px")